# COMS3007A — Machine Learning Assignment 2
## Applied ML Pipeline: Wearable Sensor Action Classification

**Data files required** (place in the `../data/` folder relative to this notebook):
- `train_data.parquet`
- `train_labels.csv`
- `test_data.parquet`

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
# Run this once if libraries are not installed
# In VS Code terminal you can also run: pip install -r ../requirements.txt
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'pyarrow', 'fastparquet', 'xgboost', 'scipy', 'seaborn', '-q'])
print('Dependencies ready ✅')

In [ ]:
# ── CELL 2: Imports ────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from numpy.fft import fft
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import xgboost as xgb

print('All imports successful ✅')

In [ ]:
# ── CELL 3: Load Data ──────────────────────────────────────────────────────────
# Data folder is ../data/ relative to this notebook (notebooks/ML__assignment2.ipynb)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
folder = os.path.normpath(os.path.join(notebook_dir, '..', 'data'))

print('Loading data from:', folder)

train_df  = pd.read_parquet(os.path.join(folder, 'train_data.parquet'))
labels_df = pd.read_csv(os.path.join(folder, 'train_labels.csv'))
test_df   = pd.read_parquet(os.path.join(folder, 'test_data.parquet'))

print('All files loaded ✅')
print('train_df shape :', train_df.shape)
print('labels_df shape:', labels_df.shape)
print('test_df shape  :', test_df.shape)
print('Label columns  :', labels_df.columns.tolist())

In [ ]:
# ── CELL 4: EDA — Basic structure ─────────────────────────────────────────────
print('=== Basic Structure ===')
print('Unique train samples:', train_df['Sample_ID'].nunique())
print('Unique test samples :', test_df['Sample_ID'].nunique())

rows_per_sample = train_df.groupby('Sample_ID').size()
print('\nRows per sample (should all be 100):')
print(rows_per_sample.describe())

print('\nClass distribution:')
print(labels_df['class_label'].value_counts().sort_index())

In [ ]:
# ── CELL 5: EDA — Missing values ──────────────────────────────────────────────
print('=== Missing values per column ===')
print(train_df.isnull().sum())
print('\nTotal missing cells:', train_df.isnull().sum().sum())

samples_with_missing = train_df.groupby('Sample_ID').apply(
    lambda g: g.isnull().any().any(), include_groups=False
).sum()
print(f'Samples with at least one missing value: {samples_with_missing}')

In [ ]:
# ── CELL 6: EDA — Visualise dropout and all signals ───────────────────────────
signal_cols = [c for c in train_df.columns if c.startswith('Signal')]

# Find a sample with dropout in Signal_F
samples_with_nan = train_df[train_df['Signal_F'].isnull()]['Sample_ID'].unique()
print(f'Samples with missing Signal_F: {len(samples_with_nan)}')
dropout_sample_id = samples_with_nan[0]
dropout_sample = train_df[train_df['Sample_ID'] == dropout_sample_id].set_index('Time_Step')

# Missing data heatmap
plt.figure(figsize=(14, 5))
sns.heatmap(dropout_sample[signal_cols].isnull(), cbar=False, yticklabels=10, cmap='Blues')
plt.title(f'Missing Data Heatmap — Sample ID {dropout_sample_id}')
plt.xlabel('Signal')
plt.ylabel('Time Step')
plt.tight_layout()
plt.show()

# All 14 signals for one sample
sample_plot = train_df[train_df['Sample_ID'] == train_df['Sample_ID'].iloc[0]]
fig, axes = plt.subplots(7, 2, figsize=(14, 18))
axes = axes.flatten()
for i, col in enumerate(signal_cols):
    axes[i].plot(sample_plot['Time_Step'], sample_plot[col])
    axes[i].set_title(col)
    axes[i].set_xlabel('Time Step')
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 7: Imputation — Linear interpolation per sample ──────────────────────
def impute_sample(group):
    sample_id = group['Sample_ID'].iloc[0]
    result = group.set_index('Time_Step')[signal_cols]\
                  .interpolate(method='linear', limit_direction='both')\
                  .reset_index()
    result['Sample_ID'] = sample_id
    return result

print('Imputing train data...')
train_imputed = train_df.groupby('Sample_ID', group_keys=False).apply(
    impute_sample, include_groups=False)

print('Imputing test data...')
test_imputed = test_df.groupby('Sample_ID', group_keys=False).apply(
    impute_sample, include_groups=False)

print('Missing in train after imputation:', train_imputed.isnull().sum().sum())
print('Missing in test after imputation :', test_imputed.isnull().sum().sum())
print('train_imputed shape:', train_imputed.shape)
print('test_imputed shape :', test_imputed.shape)

In [ ]:
# ── CELL 8: Visualise Signal_F before vs after imputation ─────────────────────
sample_before = train_df[train_df['Sample_ID'] == dropout_sample_id].set_index('Time_Step')
sample_after  = train_imputed[train_imputed['Sample_ID'] == dropout_sample_id].set_index('Time_Step')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(sample_before.index, sample_before['Signal_F'], color='steelblue', linewidth=1.5)
axes[0].set_title(f'Signal_F BEFORE — Sample ID {dropout_sample_id}\n(gap = sensor dropout)')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Signal_F value')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sample_after.index, sample_after['Signal_F'], color='green', linewidth=1.5)
axes[1].set_title(f'Signal_F AFTER — Sample ID {dropout_sample_id}\n(gap filled by interpolation)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Signal_F value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 9: Feature Engineering — Statistical features ────────────────────────
def extract_statistical_features(group):
    features = {}
    for col in signal_cols:
        values = group[col].values
        features[f'{col}_mean']     = np.mean(values)
        features[f'{col}_std']      = np.std(values)
        features[f'{col}_min']      = np.min(values)
        features[f'{col}_max']      = np.max(values)
        features[f'{col}_range']    = np.max(values) - np.min(values)
        features[f'{col}_skew']     = stats.skew(values)
        features[f'{col}_kurtosis'] = stats.kurtosis(values)
        features[f'{col}_median']   = np.median(values)
        features[f'{col}_q25']      = np.percentile(values, 25)
        features[f'{col}_q75']      = np.percentile(values, 75)
        features[f'{col}_iqr']      = np.percentile(values, 75) - np.percentile(values, 25)
    return pd.Series(features)

print('Extracting statistical features from train...')
train_stat = train_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_statistical_features, include_groups=False)
print('Extracting statistical features from test...')
test_stat  = test_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_statistical_features, include_groups=False)

print('Train stat shape:', train_stat.shape)
print('Test stat shape :', test_stat.shape)

In [ ]:
# ── CELL 10: Feature Engineering — Temporal & Frequency features ───────────────
def extract_temporal_features(group):
    features = {}
    for col in signal_cols:
        values = group[col].values
        diff1 = np.diff(values)
        features[f'{col}_diff_mean']  = np.mean(diff1)
        features[f'{col}_diff_std']   = np.std(diff1)
        features[f'{col}_diff_max']   = np.max(np.abs(diff1))
        mean_val = np.mean(values)
        features[f'{col}_crossings']  = np.sum(np.diff(np.sign(values - mean_val)) != 0)
        features[f'{col}_energy']     = np.sum(values ** 2)
        fft_vals = np.abs(fft(values))[:50]
        features[f'{col}_fft_mean']   = np.mean(fft_vals)
        features[f'{col}_fft_std']    = np.std(fft_vals)
        features[f'{col}_fft_max']    = np.max(fft_vals)
        features[f'{col}_fft_argmax'] = np.argmax(fft_vals)
    return pd.Series(features)

print('Extracting temporal & frequency features from train...')
train_temp = train_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_temporal_features, include_groups=False)
print('Extracting temporal & frequency features from test...')
test_temp  = test_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_temporal_features, include_groups=False)

print('Train temporal shape:', train_temp.shape)
print('Test temporal shape :', test_temp.shape)

In [ ]:
# ── CELL 11: Feature Engineering — Correlation features ───────────────────────
def extract_correlation_features(group):
    features = {}
    values_matrix = group[signal_cols].values
    corr_matrix   = np.corrcoef(values_matrix.T)
    idx = np.triu_indices(len(signal_cols), k=1)
    corr_values = corr_matrix[idx]
    for i, (s1, s2) in enumerate(zip(idx[0], idx[1])):
        key = f'corr_{signal_cols[s1]}_{signal_cols[s2]}'
        features[key] = corr_values[i]
    return pd.Series(features)

print('Extracting correlation features from train...')
train_corr = train_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_correlation_features, include_groups=False)
print('Extracting correlation features from test...')
test_corr  = test_imputed.groupby('Sample_ID', group_keys=False).apply(
    extract_correlation_features, include_groups=False)

print('Correlation features shape:', train_corr.shape)

In [ ]:
# ── CELL 12: Combine all features + attach labels ─────────────────────────────
train_features_v2 = pd.concat([train_stat, train_temp, train_corr], axis=1)
test_features_v2  = pd.concat([test_stat,  test_temp,  test_corr],  axis=1)

train_final_v2 = train_features_v2.reset_index().merge(labels_df, on='Sample_ID')

print('Combined train shape:', train_features_v2.shape)
print('Combined test shape :', test_features_v2.shape)
print('Train final shape   :', train_final_v2.shape)
print('\nClass distribution:')
print(train_final_v2['class_label'].value_counts().sort_index())

# Prepare X, y, X_test
feature_cols_v2 = [c for c in train_final_v2.columns if c not in ['Sample_ID', 'class_label']]
X_v2      = train_final_v2[feature_cols_v2].values
y_encoded = train_final_v2['class_label'].values - 1  # convert 1-6 to 0-5
X_test_v2 = test_features_v2.reset_index()[feature_cols_v2].values

print('\nX_v2 shape     :', X_v2.shape)
print('X_test_v2 shape:', X_test_v2.shape)

In [ ]:
# ── CELL 13: Cross-validation — local score to trust ──────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(model, X_v2, y_encoded,
                         cv=skf, scoring='f1_macro', n_jobs=-1)

print('Cross-validation Macro-F1 scores:', np.round(scores, 4))
print(f'Mean : {scores.mean():.4f}')
print(f'Std  : {scores.std():.4f}')
print('\n(This is your expected private leaderboard score — use this in your report)')

In [ ]:
# ── CELL 14: Train final model on ALL training data ────────────────────────────
final_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_v2, y_encoded)
print('Final model trained on full dataset ✅')

In [ ]:
# ── CELL 15: Generate predictions ─────────────────────────────────────────────
test_preds_encoded = final_model.predict(X_test_v2)
test_preds = test_preds_encoded + 1  # convert back 0-5 to 1-6

print('Predictions shape   :', test_preds.shape)
print('Unique classes      :', np.unique(test_preds))
print('Prediction distribution:')
for cls, count in zip(*np.unique(test_preds, return_counts=True)):
    print(f'  Class {cls}: {count} samples')

In [ ]:
# ── CELL 16: Save submission CSV ───────────────────────────────────────────────
test_sample_ids = test_features_v2.reset_index()['Sample_ID'].values

submission = pd.DataFrame({
    'Sample_ID'       : test_sample_ids,
    'Predicted_Class' : test_preds
})

# Save to submissions/ folder in the repo
submissions_folder = os.path.normpath(os.path.join(notebook_dir, '..', 'submissions'))
os.makedirs(submissions_folder, exist_ok=True)
submission_path = os.path.join(submissions_folder, 'submission.csv')
submission.to_csv(submission_path, index=False)

print('Submission saved ✅')
print(f'Path: {submission_path}')
print(f'Total predictions: {len(submission)}')
print(submission.head(10))